# 第2回：ルールで対話をつくる簡易ELIZA

昨年度 `chap02/doctor.py` を基にした学習用の簡易版です。原ELIZA全体の再実装ではありません。第2回スライド11〜14・31〜33と対応します。

**学習目標**：入力中の文字列、規則の順序、対話状態が応答にどう影響するかを説明する。人が規則を編集する操作と、自動学習を区別する。

**準備**：Python 3.10以降で動く記法です。授業のvenvをカーネルに選びます。Notebook用のipykernelが必要なら同梱のロック由来のrequirements.txtを対象venvへ導入してください。uvの利用方法はREADMEにあります。このNotebookは処理コードをすべて収録し、同梱Pythonファイルやデータファイル、外部モデル/APIを必要としません。

カーネルを再起動し、先頭から全セルを実行してください。固定入力は架空の教材用例です。対話入力待ちはありません。

In [ ]:
import sys
print("実行中のPython:", sys.executable)
print("Pythonの版:", sys.version.split()[0])


## 1. 規則と、対話の状態

`RULES` を上から調べ、最初に一致した規則を一つ使います。「が心配です」の規則は文字列を「は心配ですか？」へ置換します。該当なしは「続けてください」です。

`count` はオウム返しまでの状態、`turns` は全体の応答回数です。初期設定は `CYCLE=5`、`LIMIT=20`。昨年度の更新順を維持しているため、6・11・16回目にはキーワード規則より先にオウム返しします。

`Doctor()` で新しい状態を作れます。比較する際は同じ状態から始めましょう。以下には単体実行に必要な処理がすべてあります。

In [ ]:
"""昨年度 doctor.py を基にした学習用の簡易ELIZA。外部通信・学習は行わない。"""
import re

LIMIT = 20
CYCLE = 5
GREETING = "私はDoctor、お話を伺います"
FAREWELL = "それではそろそろ終了しましょう。おつかれさまでした。"

# 上から調べ、最初に一致した規則だけを使う。順序もプログラムの一部。
RULES = [
    ("先生", "私のことでなくあなたのことを話しましょう"),
    ("母", "あなたのお母さんについて話してください"),
    ("父", "あなたのお父さんについて話してください"),
    ("意見", "私の意見を聞きたいのですか？"),
    ("が心配です", "{question}"),
]

DEMO_INPUTS = [
    "先生に相談したいです", "母が心配です", "父について話します",
    "あなたの意見を聞きたいです", "試験が心配です", "今日は晴れです",
    "こんにちは", "先生と母について", "父と母について", "母校の話です",
    "試験が心配です", "意見と父について", "授業が心配です",
    "父が心配です", "母が心配です", "先生について", "意見があります",
    "締切が心配です", "こんにちは", "最後の入力です",
]


class Doctor:
    """応答の状態を持つ。rules・cycle・limit を変更して比較できる。"""

    def __init__(self, rules=None, cycle=CYCLE, limit=LIMIT):
        if cycle < 1 or limit < 1:
            raise ValueError("cycle と limit は1以上にしてください")
        self.rules = list(RULES if rules is None else rules)
        self.cycle = cycle
        self.limit = limit
        self.count = 0
        self.turns = 0
        self.last_rule = ""

    @property
    def finished(self):
        return self.turns >= self.limit

    def respond(self, text):
        if self.finished:
            raise StopIteration("対話は終了しました")
        # 昨年度と同じ更新順。既定では6・11・16回目にオウム返しする。
        if self.count >= self.cycle:
            response = text + " 、ですか・・・"
            self.count = 0
            self.last_rule = "オウム返し"
        else:
            response = "続けてください"
            self.last_rule = "既定応答"
            for pattern, template in self.rules:
                if re.search(pattern, text):
                    response = template.format(
                        question=text.replace("が心配です", "は心配ですか？")
                    )
                    self.last_rule = pattern
                    break
        self.count += 1
        self.turns += 1
        return response


def demo_responses(inputs=DEMO_INPUTS):
    doctor = Doctor()
    responses = []
    for text in inputs:
        if doctor.finished:
            break
        responses.append(doctor.respond(text))
    return responses



## 2. 同じ入力20件で、基準の応答を確認する

入力一覧は上の `DEMO_INPUTS` にあります。各行に応答と使った規則を表示します。CLIの `python doctor.py --demo` と同じ応答列になります。

**観察点**：「母が心配です」は「母」が優先される。6・11・16回目はオウム返し。「母校」も文字列「母」に一致する。20回で終了する。乱数はなく、文字列は完全一致で照合できます。

In [ ]:
baseline_responses = demo_responses()
doctor = Doctor()
for number, text in enumerate(DEMO_INPUTS, 1):
    response = doctor.respond(text)
    print(f"{number:02d} あなた> {text}")
    print(f"   Dr> {response}  [規則: {doctor.last_rule}]")
print("応答回数:", doctor.turns, "終了:", doctor.finished)


## 3. 入力だけ変える

同じ規則を使い、それぞれ新しい状態から始めます。次の三つの応答を比べてください。

- 「母が心配です」と「試験が心配です」は、どの規則が違うでしょうか。
- 「母校の話です」の応答は、入力の意味に合っているでしょうか。
- 一致した文字列から、その語の意味まで処理したと言えるでしょうか。

`inputs_to_try` の内容を変え、実行し直せます。

In [ ]:
inputs_to_try = ["母が心配です", "試験が心配です", "母校の話です"]
for text in inputs_to_try:
    doctor = Doctor()
    response = doctor.respond(text)
    print(text, "→", response, "[規則:", doctor.last_rule, "]")


## 4. 規則の順序だけ変える

「が心配です」を先頭へ移します。入力は同じ「母が心配です」です。変更前・変更後の両方を、新しい状態から試します。

**想定結果**：変更前は「あなたのお母さんについて話してください」、変更後は「母は心配ですか？」。複数の規則が一致しても、最初の一つだけが使われます。

In [ ]:
changed_rules = [RULES[-1]] + RULES[:-1]
text = "母が心配です"
for label, rules in [("変更前", RULES), ("変更後", changed_rules)]:
    doctor = Doctor(rules=rules)
    response = doctor.respond(text)
    print(label, response, "[規則:", doctor.last_rule, "]")


## 5. キーワードと応答を追加する

先頭に「締切」の規則を加えます。`custom_rules` のキーワードや応答文を変えて試せます。

**想定結果**：「締切が心配です」に対し、変更前は「締切は心配ですか？」、追加後は「締切までに何を進めますか？」。

これは人が規則を書き換えた結果です。プログラムが対話例から規則を自動で学んだわけではありません。

In [ ]:
custom_rules = [("締切", "締切までに何を進めますか？")] + RULES
text = "締切が心配です"
print("変更前:", Doctor().respond(text))
print("追加後:", Doctor(rules=custom_rules).respond(text))


## 6. 状態の影響を確認する

通常の規則と、短い周期 `cycle=2` を比較します。各版は新しい状態から始め、同じ「先生」を6回入力します。

**想定結果**：既定では6回目、短い周期では3・5回目がオウム返しです。同じ入力と規則でも、対話状態が違えば応答が変わります。回数の設定を変更することも、自動学習ではありません。

In [ ]:
for label, cycle in [("既定", 5), ("短い周期", 2)]:
    doctor = Doctor(cycle=cycle, limit=6)
    print(label)
    for number in range(1, 7):
        print(number, doctor.respond("先生"), "[規則:", doctor.last_rule, "]")


## 観察のまとめ

1. どの入力に、どの規則が一致したかを説明する。
2. 規則を並べ替えた場合は、入力と開始状態をそろえて比較する。
3. 文字列の一致、自然な返答、意味の理解を同じものとして扱わない。
4. 規則を編集する操作と、データを使って重みなどを調整する学習を区別する。

提出・採点の条件はこの教材で追加しません。プログラムの検証は章内で `make check`、リポジトリのルートで `make check-programs` を使います。実行済みNotebook・検証結果は `programs/outputs/` に分離し、配布するこの原稿の出力セルは空に保ちます。